In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from kneed import KneeLocator 
from sklearn.model_selection import train_test_split

In [ ]:
file_path = 'path/NOR.xlsx'
save_dir = 'path/'
os.makedirs(save_dir, exist_ok=True)

subject_id_col = 'ID'
df = pd.read_excel(file_path,sheet_name='Sheet1')
X = df.iloc[:,2:]  # 特征
y = df.iloc[:, 1]   # 标签
subject_ids = df[subject_id_col]  # 获取ID列数据

ratio = 0.2

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(X, y, subject_ids, test_size=ratio,
                                                    stratify=y, random_state =42)
df_train = pd.concat([X_train, y_train, ids_train], axis=1)
df_test = pd.concat([X_test, y_test, ids_test], axis=1)
df_train.to_excel(os.path.join(save_dir, 'train.xlsx'), index=False)
df_test.to_excel(os.path.join(save_dir, 'internal_valid.xlsx'), index=False)

In [ ]:
file_path = 'path/train.xlsx'
data_train = pd.read_excel(file_path, sheet_name='Sheet1')
save_dir = 'path/'
os.makedirs(save_dir, exist_ok=True)

wavelengths = data_train.columns[:-2]
wavelengths = np.array(wavelengths, dtype=float) 
spectra_data = data_train.iloc[:, :-2].values
labels = data_train.iloc[:, -2]  

pca = PCA(n_components=min(spectra_data.shape[0], spectra_data.shape[1]))  # 选取最大可能的主成分数量
pca.fit(spectra_data)

explained_variance = pca.explained_variance_ 
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance_ratio = np.cumsum(explained_variance_ratio)

kneedle = KneeLocator(range(1, len(cumulative_explained_variance_ratio) + 1), cumulative_explained_variance_ratio, curve='concave', direction='increasing')
optimal_components = kneedle.elbow

plt.figure(figsize=(5, 5))
plt.plot(range(1, len(cumulative_explained_variance_ratio) + 1), cumulative_explained_variance_ratio)
y_value = cumulative_explained_variance_ratio[optimal_components - 1]
plt.text(optimal_components, y_value, f'({optimal_components}, {y_value:.2f})', color='black', ha='right', va='bottom')
plt.hlines(y_value, xmin=0, xmax=optimal_components, linestyles='--', colors='C1')
plt.vlines(optimal_components, ymin=0, ymax=y_value, linestyles='--', colors='C1', label=f'optimal_component = {optimal_components}')
plt.xlabel('Number of Principal Components',fontsize=12)
plt.ylabel('Cumulative Explained Variance Ratio',fontsize=12)
plt.tick_params(axis='both', which='major', labelsize=12) 
plt.legend(loc='lower right')
plt.savefig(os.path.join(save_dir, 'cumulative_variance.pdf'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
pca = PCA(n_components=optimal_components) 
pca.fit(spectra_data)
train_data = pca.transform(spectra_data)
components = pca.components_[:optimal_components]
loadings_df = pd.DataFrame(components, columns=wavelengths, index=[f'PC {i+1}' for i in range(components.shape[0])])

In [ ]:
train_df = pd.DataFrame(data=train_data, columns=[f'PC{i+1}' for i in range(optimal_components)])
train_df = pd.concat([labels.reset_index(drop=True), train_df], axis=1)
train_df.to_excel(os.path.join(save_dir, 'train_PCA.xlsx'), index=False)

In [ ]:
valid_path = 'path/internal_valid.xlsx' # prospective_valid.xlsx & external_valid.xlsx 
valid_data = pd.read_excel(valid_path, sheet_name='Sheet1')

labels_valid = valid_data.iloc[:,-2]  
wavelengths = valid_data.columns[:-2]  
wavelengths = np.array(wavelengths, dtype=float) 
spectra_data = valid_data.iloc[:, :-2].values 
valid_data = pca.transform(spectra_data)

V1_df = pd.DataFrame(data=valid_data, columns=[f'PC{i+1}' for i in range(optimal_components)])
valid_df = pd.concat([labels_valid.reset_index(drop=True), V1_df], axis=1)
valid_df.to_excel(os.path.join(save_dir, 'internal_valid_PCA.xlsx'), index=False) # prospective_valid_PCA.xlsx & external_valid_PCA.xlsx 